# Predictive Maintenance — Feature Engineering

This notebook creates a small number of domain-relevant features from machine operating conditions to improve the representation of the data for machine learning.

## Engineered Features

1. Temperature Difference
2. Mechanical Power

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("../data/processed/cleaned_data.csv")
data.head()

,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [3]:
print("Dataset shape:", data.shape)

Dataset shape: (10000, 7)


In [4]:
print(data.columns.tolist())

['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure']


In [5]:
data["Temperature difference [K]"] = (
    data["Process temperature [K]"]
    - data["Air temperature [K]"]
)

In [6]:
data[
    [
        "Air temperature [K]",
        "Process temperature [K]",
        "Temperature difference [K]"
    ]
].head()

,Air temperature [K],Process temperature [K],Temperature difference [K]
0,298.1,308.6,10.5
1,298.2,308.7,10.5
2,298.1,308.5,10.4
3,298.2,308.6,10.4
4,298.2,308.7,10.5


In [7]:
data["Mechanical power [W]"] = (
    data["Torque [Nm]"]
    * data["Rotational speed [rpm]"]
    * 2 * np.pi / 60
)

In [8]:
data[
    [
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Mechanical power [W]"
    ]
].head()

,Rotational speed [rpm],Torque [Nm],Mechanical power [W]
0,1551,42.8,6951.590560
1,1408,46.3,6826.722724
2,1498,49.4,7749.387543
3,1433,39.5,5927.504659
4,1408,40.0,5897.816608


In [9]:
engineered_features = [
    "Temperature difference [K]",
    "Mechanical power [W]"
]

print(data[engineered_features].describe())

       Temperature difference [K]  Mechanical power [W]
count                10000.000000          10000.000000
mean                    10.000630           6279.744953
std                      1.001094           1067.418295
min                      7.600000           1148.440610
25%                      9.300000           5561.184484
50%                      9.800000           6271.027344
75%                     11.000000           7003.002724
max                     12.100000          10469.923005


In [10]:
print("Missing values:")
print(data[engineered_features].isnull().sum())

print("\nInfinite values:")
print(np.isinf(data[engineered_features]).sum())

Missing values:
Temperature difference [K]    0
Mechanical power [W]          0
dtype: int64

Infinite values:
Temperature difference [K]    0
Mechanical power [W]          0
dtype: int64


In [11]:
comparison = data.groupby("Machine failure")[
    engineered_features
].mean()

comparison

,Temperature difference [K],Mechanical power [W]
Machine failure,,
0,10.021571,6244.547534
1,9.403835,7282.819485


In [12]:
feature_columns = [
    "Type",
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]",
    "Temperature difference [K]",
    "Mechanical power [W]"
]

target_column = "Machine failure"

In [13]:
print("Number of features:", len(feature_columns))
print(feature_columns)

Number of features: 8
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature difference [K]', 'Mechanical power [W]']


In [14]:
print("=" * 50)
print("FINAL FEATURE ENGINEERING CHECK")
print("=" * 50)

print("\nDataset shape:")
print(data.shape)

print("\nFeatures:")
print(feature_columns)

print("\nTarget:")
print(target_column)

print("\nMissing values:")
print(data.isnull().sum().sum())

print("\nDuplicate rows:")
print(data.duplicated().sum())

print("\nEngineered feature summary:")
print(data[engineered_features].describe())

FINAL FEATURE ENGINEERING CHECK

Dataset shape:
(10000, 9)

Features:
['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Temperature difference [K]', 'Mechanical power [W]']

Target:
Machine failure

Missing values:
0

Duplicate rows:
0

Engineered feature summary:
       Temperature difference [K]  Mechanical power [W]
count                10000.000000          10000.000000
mean                    10.000630           6279.744953
std                      1.001094           1067.418295
min                      7.600000           1148.440610
25%                      9.300000           5561.184484
50%                      9.800000           6271.027344
75%                     11.000000           7003.002724
max                     12.100000          10469.923005


In [15]:
output_path = "../data/processed/engineered_data.csv"

data.to_csv(
    output_path,
    index=False
)

print(f"Engineered dataset saved to: {output_path}")

Engineered dataset saved to: ../data/processed/engineered_data.csv


## Feature Engineering Findings

Two domain-relevant features were created.

### Temperature Difference

The difference between process temperature and air temperature was calculated to represent the thermal gap between the machine process and its surrounding environment.

### Mechanical Power

Mechanical power was calculated from torque and rotational speed using:

Power = Torque × Angular Speed

where rotational speed in RPM was converted to radians per second.

These features provide additional representations of the machine's thermal and mechanical operating conditions. Their actual contribution to predictive performance will be evaluated during model training rather than assumed in advance.